> ### Import Library

In [ ]:
# import library pandas untuk pemrosesan tabel data
import pandas as pd
# import library numpy untuk komputasi numerik dan array
import numpy as np
# import library matplotlib untuk pengaturan layout kanvas grafik
import matplotlib.pyplot as plt
# import library seaborn untuk visualisasi distribusi data klinis
import seaborn as sns

# atur agar seluruh kolom ditampilkan tanpa dipotong titik-titik
pd.set_option("display.max_columns", None)
# atur batas tampilan baris maksimal 100 agar colab tetap ringan
pd.set_option("display.max_rows", 100)


- Read the dataset

In [ ]:
# data_a = pd.read_csv("ecommerce_sales_customer_analytics_150k.csv")
# data_b = pd.read_csv("order_items.csv")
# data_c = pd.read_csv("product_catalog.csv")

In [ ]:
# data_b.head(1)

In [ ]:
# # Merge. data_b is the base (item level). Suffixes separate overlapping columns.
# merged = (
#     data_b
#     .merge(data_a, on="order_id",   how="left",
#            suffixes=("_item", "_order"), validate="many_to_one")
#     .merge(data_c, on="product_id", how="left",
#            suffixes=("_item", "_catalog"), validate="many_to_one")
# )

# # Column order
# col_order = [
#     # order identity
#     "order_id", "order_date", "order_time", "order_status",

#     # product identity + catalog price
#     "product_id", "product_name", "product_category", "product_subcategory",
#     "brand", "supplier", "product_rating", "unit_price_catalog", "product_cost",

#     # item level metrics (data_b)
#     "quantity_item", "unit_price_item", "discount_percentage",
#     "discount_amount_item", "gross_sales_item", "tax_amount_item",
#     "shipping_cost_item", "net_sales_item", "product_cost_item", "profit_item",

#     # order level totals (data_a)
#     "quantity_order", "gross_sales_order", "discount_amount_order", "tax_amount_order",
#     "shipping_cost_order", "net_sales_order", "product_cost_order",
#     "profit_order", "profit_margin_percentage",

#     # customer
#     "customer_id", "customer_name", "customer_age", "gender",
#     "customer_segment", "customer_type",
#     "customer_city", "customer_state", "customer_country",
#     "region", "customer_postal_code",

#     # channel & marketing
#     "sales_channel", "marketing_channel", "campaign_name", "coupon_code",

#     # payment
#     "payment_method", "payment_status", "currency",

#     # shipping & return
#     "shipping_method", "warehouse", "delivery_days", "estimated_delivery_days",
#     "delivery_status", "return_status", "return_reason",

#     # review
#     "customer_rating", "review_sentiment", "customer_review",

#     # loyalty & customer value
#     "loyalty_points_earned", "loyalty_points_redeemed",
#     "customer_lifetime_value", "is_repeat_customer", "customer_order_count",
# ]

# # Append any leftover columns at the end (seharusnya kosong sekarang)
# sisa = [c for c in merged.columns if c not in col_order]
# print("Kolom sisa:", sisa)

# merged = merged[[c for c in col_order if c in merged.columns] + sisa]
# merged = merged.sort_values(["order_id", "product_id"]).reset_index(drop=True)

# merged.to_csv("E-commerce_Dataset.csv", index=False)

> ### Understanding the dataset

In [ ]:
data = pd.read_csv("E-commerce_Dataset.csv")

- Show 5 rows

In [ ]:
data.head()

- data rows and columns details

In [ ]:
data.info()

> ### Dataset Documentation

#### Overview

| Attribute | Value |
|---|---|
| Total rows | 397,569 |
| Total columns | 65 |
| Memory usage | ~194.5 MB |
| Grain | One row = one product line within one order |
| Primary key | (`order_id`, `product_id`) |

This dataset is a join of three sources:

- **Order header** (per `order_id`) — customer, payment, shipping, order-level totals
- **Order items** (per `order_id` + `product_id`) — item-level price, quantity, profit
- **Product catalog** (per `product_id`) — product name, category, brand, supplier, catalog prices

The item table is the base, so each order appears as many rows as it has distinct products.

### Suffix Convention

| Suffix | Source | Meaning |
|---|---|---|
| `_item` | Order items | Value for the specific product line |
| `_order` | Order header | Value for the entire order, repeated on every row of that order |
| *(none)* | Product catalog | Static product master attribute |
| `_x` / `_y` | Merge collision | Same-named columns from different sources |

**On `unit_price_x` and `unit_price_y`:**

- `unit_price_x` — actual price per unit in this order line (from the item table)
- `unit_price_y` — catalog list price (from the product master)

### Column Groups

####  1. Order Identity

`order_id`, `order_date`, `order_time`, `order_status`

####  2. Product Master

`product_id`, `product_name`, `product_category`, `product_subcategory`, `brand`, `supplier`, `product_rating`, `product_cost`

`product_cost` here is the catalog cost of goods, not the actual cost booked to this order.

####  3. Item-Level Metrics (`*_item`)

`quantity_item`, `discount_percentage`, `discount_amount_item`, `gross_sales_item`, `tax_amount_item`, `shipping_cost_item`, `net_sales_item`, `product_cost_item`, `profit_item`

These are the correct values to aggregate for product-level analysis (revenue by product, margin by product, discount analysis). Summing `profit_item` or `net_sales_item` across rows gives total revenue and profit.

####  4. Order-Level Metrics (`*_order`)

`quantity_order`, `gross_sales_order`, `discount_amount_order`, `tax_amount_order`, `shipping_cost_order`, `net_sales_order`, `product_cost_order`, `profit_order`, `profit_margin_percentage`

These describe the whole transaction and are repeated on every row of the same `order_id`. Do not sum these across rows — use `first()` per `order_id`, or deduplicate on `order_id` before aggregating.

####  5. Customer

`customer_id`, `customer_name`, `customer_age`, `gender`, `customer_segment`, `customer_type`, `customer_city`, `customer_state`, `customer_country`, `region`, `customer_postal_code`

####  6. Sales & Marketing

`sales_channel`, `marketing_channel`, `campaign_name`, `coupon_code`

####  7. Payment

`payment_method`, `payment_status`, `currency`

`payment_status` and `order_status` are not always aligned — an order can be `Completed` with `Pending` payment.

####  8. Shipping

`shipping_method`, `warehouse`, `delivery_days`, `estimated_delivery_days`, `delivery_status`

####  9. Returns

`return_status`, `return_reason`

####  10. Reviews

`customer_rating`, `review_sentiment`, `customer_review`

####  11. Loyalty & Customer Value

`loyalty_points_earned`, `loyalty_points_redeemed`, `customer_lifetime_value`, `is_repeat_customer`, `customer_order_count`

### Null Patterns

Nulls are not random; they correspond to events that did not happen.

| Column | Non-null | Interpretation |
|---|---|---|
| `campaign_name` | 157,810 | Null = order not tied to a named campaign |
| `coupon_code` | 79,484 | Null = no coupon used |
| `delivery_days`, `estimated_delivery_days`, `delivery_status` | 327,028 | Null = order not yet delivered |
| `customer_rating`, `review_sentiment`, `customer_review` | 327,028 | Null = no review submitted |
| `return_status`, `return_reason` | 27,209 | Non-null = returned; null = not returned |

The identical non-null count (327,028) across delivery, rating, and review fields indicates these are populated together once an order is delivered and reviewed.

### Data Types

- `object` (33) — identifiers, categorical text, dates and times stored as strings
- `float64` (24) — monetary values, percentages, ratings
- `int64` (7) — counts and points
- `bool` (1) — `is_repeat_customer`

`order_date` and `order_time` are strings, not datetimes. Convert with `pd.to_datetime` before any time-based analysis.

### Usage Notes

1. **Pick the right level before aggregating.** Use `*_item` for product analysis and `*_order` for order or customer analysis. Mixing them produces inflated totals.
2. **Deduplicate to order level** when computing customer metrics:

   ```python
   orders = df.drop_duplicates("order_id")
   ```

3. **Order-level totals should reconcile with item totals per order**, but small rounding differences exist — for example, shipping cost is sometimes allocated entirely to one line.
4. **Currency varies by row** (`USD`, `EUR`, `INR`, …). Amount columns are not normalized — filter by currency or convert before summing across regions.

> ### Data Cleaning

- Check Missing Values

In [ ]:
data.isnull().sum()

- show some rows that include missing values

In [ ]:
# columns that contain at least one null
null_cols = data.columns[data.isna().any()]
null_idx  = [data.columns.get_loc(c) for c in null_cols]

# red header only for columns with nulls
header_styles = [
    {
        "selector": f"th.col{i}",
        "props": [
            ("background-color", "red"),
            ("color", "white"),
            ("font-weight", "bold"),
        ],
    }
    for i in null_idx
]

(
    data.head(10).style
        .set_table_styles(header_styles, overwrite=False)
        .map(lambda v: "background-color: #ffd6d6" if pd.isna(v) else "", subset=null_cols)
)

- Check the missing values percentage

In [ ]:
missing = (
    pd.DataFrame({
        "missing_count": data.isna().sum(),
        "missing_pct":   (data.isna().mean() * 100).round(2),
    })
    .query("missing_count > 0")
    .sort_values("missing_pct", ascending=False)
)

missing["missing_pct"] = missing["missing_pct"].map(lambda x: f"{x:.2f}%")
missing

As we can see here, it's looks like we got pattern here because there are 5 columns that got same percentage and missing count.

Let's dig it deeper!

- Checking, is 5 columns that got missing values there have same rows too?

In [ ]:
cols = ["delivery_days", "estimated_delivery_days",
        "customer_rating", "review_sentiment", "customer_review"]
print(data[cols].isna().nunique(axis=1).eq(1).all())   # True = identical null pattern

So, we got conclusion that all of missing values based on those columns are same!

- Check the missing values based on another columns 

In [ ]:
def null_profile(df, col, driver, kind="status"):
    """Print null vs non-null breakdown for one column against a driver column."""
    flag = df[col].isna()
    print(f"\n===> {col} (null: {flag.sum():,} / {len(df):,} = {flag.mean()*100:.2f}%) ===")
    print(f"driver: {driver}")
    if kind == "status":
        out = (df.assign(flag=flag)
                 .groupby("flag")[driver]
                 .value_counts(normalize=True).unstack().mul(100).round(2))
    else:
        out = (df.assign(flag=flag)
                 .groupby("flag")[driver]
                 .agg(["mean", "median", "max"]).round(2))
    print(out)


print(f"Total missing: {data.isna().sum().sum():,} / {data.size:,} "
      f"({data.isna().sum().sum()/data.size*100:.2f}%)\n")

# hypothesis tests
null_profile(data, "delivery_days",  "order_status",      "status")
null_profile(data, "customer_rating","order_status",      "status")
null_profile(data, "return_status",  "order_status",      "status")
null_profile(data, "coupon_code",    "discount_amount_order", "numeric")
null_profile(data, "campaign_name",  "marketing_channel", "status")

# rule violations
violations = data[
    (data["coupon_code"].notna() & (data["discount_amount_order"] == 0)) |
    ((data["order_status"] == "Cancelled") & data["delivery_days"].notna()) |
    ((data["order_status"] == "Completed") & data["delivery_days"].isna()) |
    (data["return_reason"].notna() & data["return_status"].isna())
]
print(f"\nRule violations: {len(violations):,} / {len(data):,} "
      f"({len(violations)/len(data)*100:.2f}%)")

In [ ]:
ret = data[data["order_status"] == "Returned"]

ret.assign(delivery_null=ret["delivery_days"].isna()) \
   .groupby("delivery_null")["return_status"] \
   .value_counts(dropna=False).unstack(fill_value=0)

> ### Missing Values, 5 rows same

Total missing cells: 1,651,269 / 25,841,985 (6.39%). The missing values fall into three patterns.

- Structural: order not delivered or not kept (17.74%)

Columns: `delivery_days`, `estimated_delivery_days`, `customer_rating`, `review_sentiment`, `customer_review`.

All five columns are null on the exact same rows. Non-null values appear **only** on `Completed` orders.

| Group | Cancelled | Completed | Pending | Returned |
|---|---|---|---|---|
| Null | 34.18% | — | 27.25% | 38.57% |
| Non-null | — | 100.00% | — | — |

Each null group has a clear reason:

- Cancelled (34.18%): order cancelled before shipping.
- Pending (27.25%): order not yet processed.
- Returned (38.57%): all 27,209 rows carry `return_status = "Returned"`, so the return is fully recorded. `delivery_days` is null because the field tracks delivery of kept orders only. Returned orders are a separate terminal state.

Handling: keep as null. The null matches the data model, not a data quality gap.

- Structural: no return occurred (93.16%)

Columns: `return_status`, `return_reason`.

Non-null values appear **only** on `Returned` orders. Among nulls, 88.30% are `Completed` orders kept by the customer.

Handling: keep as null. Null means "not returned".

- Unexplained: requires further investigation

`coupon_code` (80.01% null). The hypothesis that null = no discount is rejected. Discount distribution is nearly identical in both groups.

| Group | Mean discount | Median discount | Max discount |
|---|---|---|---|
| Coupon present | 281.17 | 137.62 | 5,046.32 |
| Coupon null | 286.59 | 137.52 | 6,759.02 |

`campaign_name` (60.31% null). The hypothesis that null = not from a campaign is rejected. Marketing channel distribution is identical in both groups (Direct 14.79% vs 14.98%, Organic Search 20.13% vs 19.93%, and so on).

Handling: not imputed. There is no reliable driver to condition on.

- Rule violations (2.62%, 10,426 rows)

Four consistency checks were run:

- Coupon present but no discount
- Cancelled but delivered
- Completed but no delivery date
- Return reason without return status

Breakdown per rule: to be filled after the next check.

- Checking the 4 columns left

In [ ]:
# return_status & return_reason: same null rows?
print("Identical null pattern:",
      (data["return_status"].isna() == data["return_reason"].isna()).all())
print("Reason without status:",
      (data["return_reason"].notna() & data["return_status"].isna()).sum())
print("Status without reason:",
      (data["return_status"].notna() & data["return_reason"].isna()).sum())

both_nonnull = data["return_status"].notna() & data["return_reason"].notna()
print("\nBoth non-null -> order_status:")
print(data.loc[both_nonnull, "order_status"].value_counts())


# coupon_code: null driven by what?
print(f"\ncoupon_code null: {data['coupon_code'].isna().sum():,} "
      f"({data['coupon_code'].isna().mean()*100:.2f}%)")

for col in ["sales_channel", "customer_segment", "customer_type",
            "order_status", "payment_method", "currency", "marketing_channel"]:
    print(f"\n--- {col} ---")
    print(
        data.assign(flag=data["coupon_code"].isna())
            .groupby("flag")[col]
            .value_counts(normalize=True).unstack().mul(100).round(2)
    )


# campaign_name: null driven by what?
print(f"\ncampaign_name null: {data['campaign_name'].isna().sum():,} "
      f"({data['campaign_name'].isna().mean()*100:.2f}%)")

for col in ["sales_channel", "customer_segment", "customer_type",
            "order_status", "payment_method", "currency", "marketing_channel"]:
    print(f"\n--- {col} ---")
    print(
        data.assign(flag=data["campaign_name"].isna())
            .groupby("flag")[col]
            .value_counts(normalize=True).unstack().mul(100).round(2)
    )


# are coupon_code and campaign_name nulls related?
print(
    pd.crosstab(
        data["coupon_code"].isna().rename("coupon_null"),
        data["campaign_name"].isna().rename("campaign_null"),
        margins=True,
    )
)

- No observable driver: optional field (80.01% and 60.31%)

Columns: `coupon_code` (80.01% null), `campaign_name` (60.31% null).

Both columns were tested against seven complete columns. In every comparison, the null and non-null groups show nearly identical distributions. The largest deviation stays under 0.5 percentage points, which is noise level for a dataset of this size.

Max deviation per comparison:

| Column | coupon_code | campaign_name |
|---|---|---|
| sales_channel | 0.18 pp | 0.42 pp |
| customer_segment | 0.43 pp | 0.18 pp |
| customer_type | 0.08 pp | 0.09 pp |
| order_status | 0.34 pp | 0.29 pp |
| payment_method | 0.42 pp | 0.38 pp |
| currency | 0.17 pp | 0.23 pp |
| marketing_channel | 0.12 pp | 0.19 pp |

The two columns are also independent of each other:

| | campaign present | campaign null |
|---|---|---|
| coupon present | 32,044 | 47,440 |
| coupon null | 125,766 | 192,319 |

Expected count for the null+null cell under independence is 191,871. Actual is 192,319 (0.2% deviation). No association between the two null patterns.

Interpretation: nulls in both columns are not driven by any observable column. No channel, status, segment, or payment method shows a notably higher null rate. This is consistent with optional fields that are not always recorded, not with a bug or a broken process.

Handling: keep as null. There is no driver to condition on.

- Cross-field consistency check.

In [ ]:
rules = {
    "coupon_no_discount":      data["coupon_code"].notna() & (data["discount_amount_order"] == 0),
    "cancelled_but_delivered": (data["order_status"] == "Cancelled") & data["delivery_days"].notna(),
    "completed_no_delivery":   (data["order_status"] == "Completed") & data["delivery_days"].isna(),
    "return_reason_no_status": data["return_reason"].notna() & data["return_status"].isna(),
}
for name, mask in rules.items():
    print(f"{name:26s}: {mask.sum():>7,}  ({mask.mean()*100:5.2f}%)")

Four cross-field rules were tested. Each rule encodes a business constraint that must always hold:

| Rule | Constraint | Violations |
|---|---|---|
| coupon_no_discount | If `coupon_code` is present, `discount_amount_order` must be > 0 | 10,426 (2.62%) |
| cancelled_but_delivered | If `order_status` = Cancelled, `delivery_days` must be null | 0 |
| completed_no_delivery | If `order_status` = Completed, `delivery_days` must be filled | 0 |
| return_reason_no_status | If `return_reason` is filled, `return_status` must be filled | 0 |

The main question is : HOW THEY GOT COUPON BUT NO DISCOUNT??

In [ ]:
violations = data[data["coupon_code"].notna() & (data["discount_amount_order"] == 0)]
valid      = data[data["coupon_code"].notna() & (data["discount_amount_order"] > 0)]

print(f"Violations: {len(violations):,}")
print(f"Valid:      {len(valid):,}")

# order status of violations only
print("\nOrder status of violations:")
print(violations["order_status"].value_counts(normalize=True).round(3).mul(100))

All 10,426 rows are cancelled (47.0%) or returned (53.0%) orders, with zero completed orders. When an order is cancelled or returned, the discount is reversed to zero, while coupon_code remains as historical record of what the customer applied at checkout.

> ### Conclusion of All missing values
All missing values in this dataset are explained. Structural nulls correspond to business events that did not occur (no delivery, no return) and must be preserved as-is. The remaining nulls in coupon_code and campaign_name show no driver and are consistent with optional fields not always being recorded. No data quality issue was found in the missing value pattern or in cross-field consistency.

> ### Check Duplicated data

In [ ]:
data.duplicated().sum()

There's no duplicated data

> ### Check data type

In [ ]:
data.dtypes

- Change the data type for date columns

In [ ]:
data["order_date"] = pd.to_datetime(data["order_date"], format="%Y-%m-%d")

In [ ]:
data.dtypes

> ### EDA

In [ ]:
data.head()

- ### Data Preparation for EDA

Before running the exploratory analysis, the dataset is prepared in three steps to ensure accurate aggregations:

1. **Order-Level Deduplication (`orders`):**
   Because the base table is at the item level (one row per product within an order), order-level attributes repeat across multi-item transactions. For example, if a customer buys 3 items (Coffee, Sugar, and Tea) in a single checkout (`ORD-01`) totaling **$45.00**:

   | Row | `order_id` | `product_name` | `net_sales_item` | `net_sales_order` | `payment_method` |
   |---|---|---|---|---|---|
   | 1 | `ORD-01` | Coffee | $20.00 | **$45.00** | **GoPay** |
   | 2 | `ORD-01` | Sugar | $15.00 | **$45.00** | **GoPay** |
   | 3 | `ORD-01` | Tea | $10.00 | **$45.00** | **GoPay** |

   - Summing `net_sales_order` directly across all rows would inflate revenue to **$135.00** ($45 × 3), and `GoPay` would be counted 3 times for a single transaction.
   - Deduplicating by `order_id` (`orders`) isolates one row per transaction for accurate order, payment, and customer metrics, while the full table (`data`) is retained for product-level (`*_item`) analysis.

2. **Temporal Feature Extraction:**
   Extracting `order_year_month`, `order_day_name`, and `order_hour` enables time-series growth and peak shopping window analysis.

3. **Completed Transaction Filtering (`completed_orders` & `completed_items`):**
   Isolating `Completed` orders ensures that revenue, profit, and margin calculations reflect realized sales rather than cancelled, pending, or returned transactions.

In [ ]:
# Create order-level dataframe to avoid double-counting order metrics
orders = data.drop_duplicates(subset="order_id").copy()

# Extract time features for temporal analysis
orders["order_year_month"] = orders["order_date"].dt.to_period("M").astype(str)
orders["order_day_name"] = orders["order_date"].dt.day_name()
orders["order_hour"] = pd.to_datetime(orders["order_time"], format="%H:%M:%S").dt.hour

# Filter completed transactions for accurate revenue/profit analysis
completed_orders = orders[orders["order_status"] == "Completed"].copy()
completed_items = data[data["order_status"] == "Completed"].copy()

print(f"Total unique orders: {len(orders):,}")
print(f"Total completed orders: {len(completed_orders):,}")
print(f"Total completed items: {len(completed_items):,}")

- ### 1. Order Funnel & Payment Friction Analysis

**Objective:**
Evaluate overall order conversion and identify revenue leakage across transaction statuses (`Completed`, `Cancelled`, `Returned`, and `Pending`), while pinpointing which payment methods experience the highest cancellation or failure rates.

**Key Questions:**
- What proportion of placed orders successfully convert into `Completed` transactions?
- How much transaction volume is lost to cancellations and returns?
- Are specific payment methods (e.g., Cash on Delivery, Buy Now Pay Later, Debit/Credit Card) associated with higher cancellation or payment failure rates?

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(15, 13))
total_orders = len(orders)

# Top Plot: Order Status Distribution
status_counts = orders["order_status"].value_counts()
sns.barplot(x=status_counts.index, y=status_counts.values, ax=axes[0], hue=status_counts.index, palette="viridis", legend=False, width=0.5)
axes[0].set_title("Order Status Distribution", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Number of Orders")
axes[0].set_xlabel("Order Status")
axes[0].set_ylim(0, status_counts.max() * 1.15)

for p in axes[0].patches:
    val = int(p.get_height())
    pct = (val / total_orders) * 100
    axes[0].annotate(f"{val:,} ({pct:.1f}%)", (p.get_x() + p.get_width() / 2., val),
                     ha="center", va="bottom", fontsize=10, fontweight="bold")

# Bottom Plot: Payment Method Popularity Stacked by Order Status with Color-Matched Callout Boxes
pm_order = orders["payment_method"].value_counts().index
status_order = ["Completed", "Returned", "Cancelled", "Pending"]
status_colors = {
    "Completed": "#4c72b0",
    "Returned": "#dd8452",
    "Cancelled": "#c44e52",
    "Pending": "#8172b3"
}

pm_ct = pd.crosstab(orders["payment_method"], orders["order_status"]).loc[pm_order, status_order]
pm_pct = pm_ct.div(pm_ct.sum(axis=1), axis=0) * 100

bar_width = 0.35
pm_ct.plot(
    kind="bar",
    stacked=True,
    ax=axes[1],
    color=[status_colors[s] for s in status_order],
    width=bar_width
)
axes[1].set_title("Payment Method Popularity by Order Status", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Number of Orders")
axes[1].set_xlabel("Payment Method")
axes[1].set_ylim(0, pm_ct.sum(axis=1).max() * 1.18)
axes[1].set_xlim(-0.35, len(pm_order) - 0.15)
axes[1].tick_params(axis="x", rotation=90)
axes[1].legend(title="Order Status", bbox_to_anchor=(1.01, 1), loc="upper left")

# Vertical offsets for the 3 thin segments so callout boxes stack neatly
callout_offsets = {"Returned": -1500, "Cancelled": 1200, "Pending": 4000}

for idx, payment in enumerate(pm_order):
    row_total = pm_ct.loc[payment].sum()
    row_share = (row_total / total_orders) * 100

    # Total count & share on top of the bar
    axes[1].text(idx, row_total + (pm_ct.sum(axis=1).max() * 0.015),
                 f"{row_total:,} ({row_share:.1f}%)",
                 ha="center", va="bottom", fontsize=9, fontweight="bold")

    cum_height = 0
    for status in status_order:
        val = pm_ct.loc[payment, status]
        pct = pm_pct.loc[payment, status]
        mid_y = cum_height + val / 2
        box_color = status_colors[status]

        if status == "Completed":
            # Large segment: keep label inside the bar
            axes[1].text(idx, mid_y, f"{val:,}\n({pct:.1f}%)",
                         ha="center", va="center", fontsize=8.5, color="white", fontweight="bold")
        else:
            # Thin segments: color-matched leader line & callout box with white text
            axes[1].annotate(
                f"{val:,} ({pct:.1f}%)",
                xy=(idx + bar_width / 2, mid_y),
                xytext=(idx + bar_width / 2 + 0.06, mid_y + callout_offsets[status]),
                ha="left", va="center",
                fontsize=7.5, color="white", fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.22", fc=box_color, ec="none", alpha=0.95),
                arrowprops=dict(arrowstyle="-", color=box_color, lw=1.2)
            )
        cum_height += val

plt.tight_layout()
plt.show()

----
#### Key Findings: Order Funnel & Payment Method Analysis

1. **Overall Order Conversion & Leakage (Total Unique Orders: 138,116)**
   - **Strong Fulfillment Rate:** **82.2%** of all placed orders (**113,559 orders**) were successfully `Completed`.
   - **17.8% Order Leakage:** The remaining **24,557 orders** did not convert into realized revenue, split across **Returned** at **6.9%** (**9,462 orders**), **Cancelled** at **6.1%** (**8,398 orders**), and **Pending** at **4.8%** (**6,697 orders**).

2. **Payment Method Preference**
   - **Card Payments Dominate Half of All Volume:** **Credit Card** is the top payment method with **41,460 orders (30.0%)**, followed by **Debit Card** at **27,635 orders (20.0%)**. Together, they account for exactly **50.0%** (**69,095 orders**) of all transactions.
   - **Secondary & Niche Channels:** **PayPal** ranks third at **20,750 orders (15.0%)**, followed by **Digital Wallet** (**16,382 orders | 11.9%**), **Cash on Delivery** (**13,863 orders | 10.0%**), **Buy Now Pay Later** (**11,061 orders | 8.0%**), and **Bank Transfer** as the least used method (**6,965 orders | 5.0%**).

3. **Order Status Consistency Across Payment Methods**
   - **Uniform Completion & Friction Rates:** Completion rates remain highly consistent across all seven payment methods, ranging narrowly between **81.8%** (`PayPal`) and **82.8%** (`Digital Wallet`).
   - **No Single Payment Bottleneck:** Cancellation rates (**6.0% to 6.4%**) and return rates (**6.5% to 7.1%**) show minimal variance across payment types. This indicates that cancellations and returns are driven by broader post-checkout factors (such as product expectations or fulfillment delays) rather than friction from a specific payment method.

 ----

- ### 2. Currency Distribution & Monthly Revenue Trend (Time-Series)

**Objective:**
Examine the currency breakdown across completed orders to establish a consistent monetary baseline, then track monthly Net Sales and Profit performance over time to identify growth patterns, seasonality, or anomalies.

**Key Questions:**
- Which currencies dominate completed transactions, and what share of order volume does the primary currency represent?
- How have monthly Net Sales and Profit evolved over time within the primary currency (`USD`)?
- Are there noticeable seasonal spikes, dips, or consistent profit margins across months?

In [ ]:
from matplotlib.ticker import FuncFormatter

fig, axes = plt.subplots(2, 1, figsize=(16, 13))
total_comp = len(completed_orders)

# Top Plot: Completed Orders by Currency
curr_counts = completed_orders["currency"].value_counts()
sns.barplot(
    x=curr_counts.index,
    y=curr_counts.values,
    ax=axes[0],
    hue=curr_counts.index,
    palette="Blues_r",
    legend=False,
    width=0.5
)
axes[0].set_title("Completed Orders by Currency", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Completed Orders")
axes[0].set_xlabel("Currency")
axes[0].set_ylim(0, curr_counts.max() * 1.15)
axes[0].tick_params(axis="x", rotation=90)

for p in axes[0].patches:
    val = int(p.get_height())
    pct = (val / total_comp) * 100
    axes[0].annotate(
        f"{val:,} ({pct:.1f}%)",
        (p.get_x() + p.get_width() / 2., val),
        ha="center",
        va="bottom",
        fontsize=9.5,
        fontweight="bold"
    )

# Bottom Plot: Monthly Net Sales & Profit Trend (USD - Annotated Peaks & Lows)
usd_orders = completed_orders[completed_orders["currency"] == "USD"]
monthly_usd = usd_orders.groupby("order_year_month").agg(
    orders_count=("order_id", "count"),
    net_sales=("net_sales_order", "sum"),
    profit=("profit_order", "sum")
).reset_index()

total_usd_sales = monthly_usd["net_sales"].sum()
total_usd_profit = monthly_usd["profit"].sum()
avg_usd_sales = monthly_usd["net_sales"].mean()
avg_usd_profit = monthly_usd["profit"].mean()

axes[1].plot(
    monthly_usd["order_year_month"],
    monthly_usd["net_sales"],
    marker="o",
    label="Net Sales (USD)",
    color="#1f77b4",
    linewidth=2
)
axes[1].plot(
    monthly_usd["order_year_month"],
    monthly_usd["profit"],
    marker="s",
    label="Profit (USD)",
    color="#2ca02c",
    linewidth=2
)

# Add horizontal average reference lines
axes[1].axhline(avg_usd_sales, color="#1f77b4", linestyle="--", alpha=0.5, label=f"Avg Net Sales (${avg_usd_sales/1e6:.2f}M)")
axes[1].axhline(avg_usd_profit, color="#2ca02c", linestyle="--", alpha=0.5, label=f"Avg Profit (${avg_usd_profit/1e6:.2f}M)")

# Annotate Highest Peak and Lowest Dip for Net Sales
max_s_idx = monthly_usd["net_sales"].idxmax()
min_s_idx = monthly_usd["net_sales"].idxmin()

for idx, label_prefix, offset_y in [(max_s_idx, "Peak", 14), (min_s_idx, "Low", -24)]:
    val = monthly_usd.loc[idx, "net_sales"]
    pct = (val / total_usd_sales) * 100
    axes[1].annotate(
        f"{label_prefix}: ${val/1e6:.2f}M ({pct:.1f}%)",
        xy=(idx, val),
        xytext=(0, offset_y),
        textcoords="offset points",
        ha="center",
        fontsize=8.5,
        fontweight="bold",
        color="white",
        bbox=dict(boxstyle="round,pad=0.25", fc="#1f77b4", ec="none")
    )

# Annotate Highest Peak and Lowest Dip for Profit
max_p_idx = monthly_usd["profit"].idxmax()
min_p_idx = monthly_usd["profit"].idxmin()

for idx, label_prefix, offset_y in [(max_p_idx, "Peak", 14), (min_p_idx, "Low", -24)]:
    val = monthly_usd.loc[idx, "profit"]
    pct = (val / total_usd_profit) * 100
    axes[1].annotate(
        f"{label_prefix}: ${val/1e3:.0f}K ({pct:.1f}%)",
        xy=(idx, val),
        xytext=(0, offset_y),
        textcoords="offset points",
        ha="center",
        fontsize=8.5,
        fontweight="bold",
        color="white",
        bbox=dict(boxstyle="round,pad=0.25", fc="#2ca02c", ec="none")
    )

axes[1].set_title("Monthly Net Sales & Profit Trend (USD - Completed Orders)", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Amount (USD)")
axes[1].set_xlabel("Year-Month")
axes[1].set_ylim(monthly_usd["profit"].min() * 0.65, monthly_usd["net_sales"].max() * 1.15)
axes[1].yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"${x/1e6:.2f}M"))
axes[1].tick_params(axis="x", rotation=90)
axes[1].legend(title="Metric", loc="upper left", ncol=2)
axes[1].grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

In [ ]:
pd.crosstab(completed_orders["currency"], completed_orders["customer_country"])

---
#### Key Findings: Currency Distribution & Monthly Revenue Trend

1. **Market Concentration & 1-to-1 Country-Currency Mapping (Total Completed Orders: 113,559)**
   - **US Market Dominance:** **USD (USA)** accounts for **59.9%** (**67,978 orders**) of all completed transactions, making it the primary market baseline for financial analysis.
   - **Regional Breakdown:** The remaining **40.1%** is distributed across **GBP (UK)** at **14.7%** (**16,652 orders**), **EUR (Germany)** at **8.4%** (**9,483 orders**), **CAD (Canada)** at **5.0%** (**5,725 orders**), **AUD (Australia)** at **4.9%** (**5,548 orders**), **INR (India)** at **4.2%** (**4,718 orders**), and **AED (UAE)** at **3.0%** (**3,455 orders**).
   - **Strict Regional Currency Mapping:** Cross-tabulation confirms a strict 1-to-1 mapping between customer country and transaction currency, meaning each regional market transacts exclusively in its local currency.

2. **Strong Year-End Seasonality in USD Net Sales**
   - **Consistent Q4 Surge:** Across all five years (2021 to 2025), USD Net Sales average **$1.39M per month** but consistently spike during November and December to exceed **$2.00M**, reaching an all-time peak of **$2.28M (2.7% of total 5-year USD sales)** in **December 2023**.
   - **Recurring February Slump:** Sales consistently drop to their annual low every February, hitting an all-time low of **$0.88M (1.1%)** in **February 2021**.

3. **Profit Margin Compression During Peak Sales Months**
   - **Flat Profit vs. Spiking Revenue:** While monthly USD Profit averages **$0.55M** (peaking at **$697K | 2.1%** in **August 2021** and dipping to **$390K | 1.2%** in **February 2021**), it remains relatively flat between **$550K and $620K** during the November and December revenue surges.
   - **Business Implication:** Because profit does not scale proportionally with the Q4 revenue spike, year-end sales volume is likely driven by heavy promotional discounting or higher fulfillment costs that compress profit margins.

---

- ### 3. Product Category Performance & Profitability Analysis (Item-Level)

**Objective:**
Evaluate item-level revenue, profit contribution, and profit margins across product categories within the primary currency (`USD`) to identify top revenue drivers and high-margin categories.

**Key Questions:**
- Which product categories generate the largest share of USD Net Sales?
- Do the highest-revenue categories also deliver the highest total profit and profit margins?
- Are there high-volume categories suffering from thin profit margins due to higher product costs or discounting?

In [ ]:
from matplotlib.ticker import FuncFormatter

# Filter completed items in primary currency (USD)
usd_items = completed_items[completed_items["currency"] == "USD"].copy()

cat_perf = usd_items.groupby("product_category").agg(
    net_sales=("net_sales_item", "sum"),
    profit=("profit_item", "sum"),
    total_qty=("quantity_item", "sum"),
    avg_discount_pct=("discount_percentage", lambda x: (x * 100).mean())
).sort_values("net_sales", ascending=False)

cat_perf["sales_share_pct"] = (cat_perf["net_sales"] / cat_perf["net_sales"].sum()) * 100
cat_perf["profit_share_pct"] = (cat_perf["profit"] / cat_perf["profit"].sum()) * 100
cat_perf["margin_pct"] = (cat_perf["profit"] / cat_perf["net_sales"]) * 100

fig, axes = plt.subplots(2, 1, figsize=(15, 13))

# Top Plot: Net Sales by Product Category (Total USD + Share %)
sns.barplot(
    x=cat_perf.index,
    y=cat_perf["net_sales"],
    ax=axes[0],
    hue=cat_perf.index,
    palette="crest",
    legend=False,
    width=0.55
)
axes[0].set_title("USD Net Sales by Product Category (Completed Items)", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Net Sales (USD)")
axes[0].set_xlabel("Product Category")
axes[0].set_ylim(0, cat_perf["net_sales"].max() * 1.18)
axes[0].yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"${x/1e6:.1f}M"))
axes[0].tick_params(axis="x", rotation=90)

for idx, p in enumerate(axes[0].patches):
    val = p.get_height()
    pct = cat_perf["sales_share_pct"].iloc[idx]
    axes[0].annotate(
        f"${val/1e6:.2f}M\n({pct:.1f}%)",
        (p.get_x() + p.get_width() / 2., val),
        ha="center",
        va="bottom",
        fontsize=8.5,
        fontweight="bold"
    )

# Bottom Plot: Total Profit & Profit Margin % by Category
sns.barplot(
    x=cat_perf.index,
    y=cat_perf["profit"],
    ax=axes[1],
    hue=cat_perf.index,
    palette="flare",
    legend=False,
    width=0.55
)
axes[1].set_title("USD Total Profit & Profit Margin % by Product Category", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Total Profit (USD)")
axes[1].set_xlabel("Product Category")
axes[1].set_ylim(0, cat_perf["profit"].max() * 1.20)
axes[1].yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"${x/1e6:.1f}M"))
axes[1].tick_params(axis="x", rotation=90)

for idx, p in enumerate(axes[1].patches):
    val = p.get_height()
    margin = cat_perf["margin_pct"].iloc[idx]
    axes[1].annotate(
        f"${val/1e6:.2f}M\n(Margin: {margin:.1f}%)",
        (p.get_x() + p.get_width() / 2., val),
        ha="center",
        va="bottom",
        fontsize=8,
        fontweight="bold"
    )

plt.tight_layout()
plt.show()

# Display detailed category summary table
display_cat = cat_perf.copy()
display_cat["net_sales"] = display_cat["net_sales"].map("${:,.2f}".format)
display_cat["profit"] = display_cat["profit"].map("${:,.2f}".format)
display_cat["total_qty"] = display_cat["total_qty"].map("{:,}".format)
display_cat["sales_share_pct"] = display_cat["sales_share_pct"].map("{:.2f}%".format)
display_cat["profit_share_pct"] = display_cat["profit_share_pct"].map("{:.2f}%".format)
display_cat["margin_pct"] = display_cat["margin_pct"].map("{:.2f}%".format)
display_cat["avg_discount_pct"] = display_cat["avg_discount_pct"].map("{:.2f}%".format)

print("Product Category Performance Summary (USD):")
display(display_cat)

---
#### Key Findings: Product Category Performance & Profitability (USD)

1. **Top Revenue Drivers (Total USD Net Sales: ~$90.88M across 15 Categories)**
   - **Top 3 Categories Generate Nearly Half of All Sales:** **Electronics** leads with **$19.34M (21.3%)**, followed by **Jewelry** at **$12.01M (13.2%)** and **Home Appliances** at **$11.90M (13.1%)**. Together, these top three categories account for **$43.25M (47.6%)** of total USD net sales.
   - **Mid-Tier & Long-Tail Categories:** **Automotive** (**$7.76M | 8.5%**) and **Sports & Outdoors** (**$7.39M | 8.1%**) form a solid second tier, while **Grocery** (**$1.42M | 1.6%**) and **Books & Media** (**$1.34M | 1.5%**) contribute the smallest revenue shares.

2. **Profit Ranking Shifts Due to Margin Differences (Total USD Profit: ~$36.10M)**
   - **Home Appliances Outperforms Jewelry in Total Profit:** Although **Jewelry** generates higher net sales (**$12.01M**) than **Home Appliances** (**$11.90M**), **Home Appliances** delivers higher total profit (**$4.54M** vs. **$4.45M**) due to its stronger profit margin (**38.1%** vs. **37.1%**).
   - **Sports & Outdoors Beats Automotive in Profit:** Similarly, **Sports & Outdoors** generates **$3.16M** in profit (at a **42.8%** margin), surpassing **Automotive** (**$2.79M** at a **35.9%** margin) despite trailing in net sales.
   - **Office Supplies Outperforms Health & Wellness in Profit:** **Office Supplies** achieves **$2.08M** in profit (**49.0%** margin), overtaking **Health & Wellness** (**$1.89M** at a **43.3%** margin).

3. **Inverse Relationship Between Revenue Volume and Profit Margin**
   - **High Volume, Low Margin:** **Electronics** is the single largest profit contributor in absolute dollars (**$5.90M**), yet it records the lowest profit margin across all categories at **30.5%** (followed by **Automotive** at **35.9%**).
   - **Low Volume, High Margin:** Smaller-revenue categories exhibit the highest profitability efficiency, led by **Grocery** (**52.6%** margin), **Beauty & Personal Care** (**49.7%** margin), **Books & Media** (**49.5%** margin), and **Office Supplies** (**49.0%** margin).
   - **Business Implication:** Bundling or cross-selling high-margin categories (such as Beauty & Personal Care, Office Supplies, or Books & Media) alongside high-traffic Electronics orders can significantly lift overall basket profitability without relying solely on heavy discounting.
---

- ### 4. Customer Demographics, Segments & Sales Channel Analysis

**Objective:**
Analyze the composition of completed orders across customer segments, age groups, and sales channels, while evaluating customer retention (`is_repeat_customer`) and spending behavior (`USD` Average Order Value and Customer Lifetime Value).

**Key Questions:**
- Which customer segments (`Consumer`, `Business`, `Premium`, `VIP`) and age groups generate the highest order volume?
- Which sales channels (`Website`, `Mobile App`, `Marketplace`, etc.) drive the majority of completed transactions?
- How heavily does the business rely on repeat customers compared to first-time buyers, and which segments hold the highest Average Order Value (AOV)?

In [ ]:
# Create age groups on completed_orders
bins = [17, 25, 35, 45, 55, 65, 100]
labels = ["18-25", "26-35", "36-45", "46-55", "56-65", "65+"]
completed_orders["age_group"] = pd.cut(completed_orders["customer_age"], bins=bins, labels=labels)
total_comp = len(completed_orders)

# --- Figure 1: Completed Orders by Customer Segment ---
plt.figure(figsize=(11, 5.5))
seg_counts = completed_orders["customer_segment"].value_counts()
ax1 = sns.barplot(x=seg_counts.index, y=seg_counts.values, hue=seg_counts.index, palette="Set2", legend=False, width=0.45)
plt.title("Completed Orders by Customer Segment", fontsize=14, fontweight="bold")
plt.ylabel("Completed Orders")
plt.xlabel("Customer Segment")
plt.ylim(0, seg_counts.max() * 1.15)
plt.xticks(rotation=90)

for p in ax1.patches:
    val = int(p.get_height())
    pct = (val / total_comp) * 100
    ax1.annotate(f"{val:,} ({pct:.1f}%)", (p.get_x() + p.get_width() / 2., val),
                 ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

# --- Figure 2: Completed Orders by Age Group ---
plt.figure(figsize=(11, 5.5))
age_counts = completed_orders["age_group"].value_counts().reindex(labels)
ax2 = sns.barplot(x=age_counts.index, y=age_counts.values, hue=age_counts.index, palette="Blues_d", legend=False, width=0.5)
plt.title("Completed Orders by Age Group", fontsize=14, fontweight="bold")
plt.ylabel("Completed Orders")
plt.xlabel("Age Group")
plt.ylim(0, age_counts.max() * 1.15)
plt.xticks(rotation=90)

for p in ax2.patches:
    val = int(p.get_height())
    pct = (val / total_comp) * 100
    ax2.annotate(f"{val:,} ({pct:.1f}%)", (p.get_x() + p.get_width() / 2., val),
                 ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

# --- Figure 3: Completed Orders by Sales Channel ---
plt.figure(figsize=(11, 5.5))
ch_counts = completed_orders["sales_channel"].value_counts()
ax3 = sns.barplot(x=ch_counts.index, y=ch_counts.values, hue=ch_counts.index, palette="Purples_r", legend=False, width=0.45)
plt.title("Completed Orders by Sales Channel", fontsize=14, fontweight="bold")
plt.ylabel("Completed Orders")
plt.xlabel("Sales Channel")
plt.ylim(0, ch_counts.max() * 1.15)
plt.xticks(rotation=90)

for p in ax3.patches:
    val = int(p.get_height())
    pct = (val / total_comp) * 100
    ax3.annotate(f"{val:,} ({pct:.1f}%)", (p.get_x() + p.get_width() / 2., val),
                 ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.show()

---
#### Key Findings: Customer Demographics, Segments & Sales Channel

1. **Customer Segment Distribution (Total Completed Orders: 113,559)**
   - **Consumer Segment Leads by Over Half:** **Consumer** accounts for **54.8%** (**62,211 orders**) of all completed transactions, more than double the second-largest segment.
   - **Premium, VIP & Business Tiers:** **Premium** represents **25.1%** (**28,544 orders**), while **VIP** (**11,506 orders | 10.1%**) and **Business** (**11,298 orders | 9.9%**) each contribute roughly one-tenth of total order volume.

2. **Broad & Even Age Distribution Across Adults**
   - **Consistent Buying Across Working-Age Groups:** Order volume is remarkably balanced across adult age brackets, led by **26-35** at **17.8%** (**20,188 orders**), **36-45** at **17.7%** (**20,100 orders**), **56-65** at **17.5%** (**19,863 orders**), and **46-55** at **17.4%** (**19,706 orders**).
   - **Younger & Senior Brackets:** The **65+** group still accounts for a substantial **15.6%** (**17,678 orders**), while **18-25** represents the smallest share at **14.1%** (**16,024 orders**).

3. **Digital-First Sales Channels Dominate Transactions**
   - **Mobile App & Website Capture Three-Quarters of Orders:** **Mobile App** is the top sales channel with **45,505 orders (40.1%)**, followed closely by **Website** at **39,606 orders (34.9%)**. Combined, owned digital platforms drive **75.0%** (**85,111 orders**) of all completed orders.
   - **Third-Party & Social Channels:** **Marketplace** contributes **15.0%** (**16,987 orders**), and **Social Media** accounts for **10.1%** (**11,461 orders**).
---

- ### 5. Operations, Logistics & Return Root-Cause Analysis

**Objective:**
Evaluate fulfillment efficiency across shipping methods and warehouses, measure the impact of delivery delays on customer satisfaction (`customer_rating`), and identify root causes behind product returns (`return_reason`).

**Key Questions:**
- How reliable are different shipping methods in meeting estimated delivery timelines (`Early`, `On Time`, vs. `Late`)?
- How severely do late deliveries impact `customer_rating` and delivery duration variance (`delivery_days` vs. `estimated_delivery_days`)?
- Which product categories and return reasons account for the highest volume of `Returned` orders?

In [ ]:
# Calculate delivery status percentage by shipping method
ship_ct = pd.crosstab(completed_orders["shipping_method"], completed_orders["delivery_status"])
ship_pct = ship_ct.div(ship_ct.sum(axis=1), axis=0) * 100
status_order = ship_pct.columns.tolist()

# Assign clean semantic colors automatically based on status name
palette_map = {}
for s in status_order:
    s_lower = str(s).lower()
    if "early" in s_lower:
        palette_map[s] = "#2A9D8F"
    elif "time" in s_lower:
        palette_map[s] = "#457B9D"
    else:
        palette_map[s] = "#E63946"

bar_colors = [palette_map[s] for s in status_order]

# Plot stacked horizontal bar chart
ax = ship_pct.plot(
    kind="barh",
    stacked=True,
    figsize=(11, 5),
    color=bar_colors,
    width=0.55
)

plt.title("Delivery Status by Shipping Method", fontsize=13, fontweight="bold")
plt.xlabel("Percentage (%)")
plt.ylabel("Shipping Method")
plt.xlim(0, 100)
plt.legend(title="Delivery Status", bbox_to_anchor=(1.01, 1), loc="upper left")

# Add count and percentage labels inside bars
for i, method in enumerate(ship_pct.index):
    cum_pct = 0
    for status in status_order:
        pct = ship_pct.loc[method, status]
        cnt = ship_ct.loc[method, status]
        if pct >= 5:
            ax.text(cum_pct + pct / 2, i, f"{cnt:,} ({pct:.1f}%)",
                    ha="center", va="center", fontsize=8.5, color="white", fontweight="bold")
        cum_pct += pct

plt.tight_layout()
plt.show()

# Calculate average customer rating by delivery status
deliv_rating = completed_orders.groupby("delivery_status").agg(
    count=("order_id", "count"),
    mean_rating=("customer_rating", "mean")
).reindex(status_order)
total_deliv = deliv_rating["count"].sum()

# Plot average rating bar chart
plt.figure(figsize=(10, 5))
ax2 = sns.barplot(
    x=deliv_rating.index,
    y=deliv_rating["mean_rating"],
    hue=deliv_rating.index,
    palette=palette_map,
    legend=False,
    width=0.45
)
plt.title("Average Customer Rating by Delivery Status", fontsize=13, fontweight="bold")
plt.xlabel("Delivery Status")
plt.ylabel("Average Rating (1 to 5)")
plt.ylim(0, 5.8)

# Add rating, order count, and percentage labels on top of bars
for i, p in enumerate(ax2.patches):
    rating_v = p.get_height()
    cnt_v = deliv_rating["count"].iloc[i]
    pct_v = (cnt_v / total_deliv) * 100
    ax2.annotate(f"{rating_v:.2f} / 5.0\n({cnt_v:,} | {pct_v:.1f}%)",
                 (p.get_x() + p.get_width() / 2., rating_v),
                 ha="center", va="bottom", fontsize=9, fontweight="bold")

plt.tight_layout()
plt.show()

# Create return reason matrix by product category
returned_items = data[data["order_status"] == "Returned"].copy()
ret_matrix = pd.crosstab(returned_items["product_category"], returned_items["return_reason"])
ret_matrix_pct = ret_matrix.div(ret_matrix.sum(axis=1), axis=0) * 100
annot_matrix = ret_matrix.astype(str) + "\n(" + ret_matrix_pct.round(1).astype(str) + "%)"

# Plot return root-cause heatmap
plt.figure(figsize=(13, 7.5))
sns.heatmap(
    ret_matrix_pct,
    annot=annot_matrix,
    fmt="",
    cmap="Reds",
    cbar_kws={"label": "Share Within Category (%)"},
    linewidths=1.5,
    linecolor="white"
)
plt.title("Return Reason by Product Category", fontsize=13, fontweight="bold")
plt.xlabel("Return Reason")
plt.ylabel("Product Category")
plt.xticks(rotation=90)
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

---
#### Key Findings: Operations, Logistics & Return Root-Cause Analysis

1. **Consistent Delivery Performance Across All Shipping Methods (Total Completed Orders: 113,559)**
   - **Uniform On-Time & Early Rates:** Delivery reliability is nearly identical regardless of the shipping tier selected. **On Time** deliveries account for **76.3% to 76.8%** across all four methods, **Early** deliveries account for **8.5% to 8.8%**, and **Delayed** orders consistently stay between **14.5% and 15.0%**.
   - **Volume Breakdown by Method:** **Standard** handles the largest shipment volume (**50,924 orders** total: **39,022 On Time**, **7,589 Delayed**, **4,313 Early**), followed by **Express** (**28,320 orders**), **Economy** (**22,855 orders**), and **Same Day** (**11,460 orders**). Because delay rates do not improve on faster tiers (`Same Day` at **14.5%** and `Express` at **15.0%** vs. `Standard` at **14.9%**), fulfillment bottlenecks likely occur at warehouse dispatch rather than transit speed.

2. **Delayed Deliveries Significantly Hurt Customer Satisfaction**
   - **Clear Rating Drop on Late Orders:** Orders delivered **Early** achieve the highest average customer rating at **4.02 / 5.0** (**9,757 orders | 8.6%**), followed by **On Time** deliveries at **3.73 / 5.0** (**86,916 orders | 76.5%**).
   - **0.51-Star Penalty for Delays:** When an order is **Delayed** (**16,886 orders | 14.9%**), the average customer rating drops to **3.22 / 5.0**, representing a **0.51-point drop** compared to `On Time` orders and a **0.80-point drop** compared to `Early` deliveries.

3. **Evenly Distributed Return Reasons Across Product Categories**
   - **No Single Dominant Return Cause:** Across all 15 product categories, return reasons are distributed fairly evenly between **10.5% and 14.8%**, indicating systemic post-purchase friction rather than a defect isolated to one specific category.
   - **Notable Category Hotspots:** The highest within-category return shares appear in **Books & Media** for `Other` (**296 items | 14.8%**), **Baby & Kids** for `Other` (**236 items | 14.2%**), **Toys & Games** for `Late Delivery` (**249 items | 14.2%**), **Fashion** for `Defective Product` (**214 items | 14.1%**), and **Home Appliances** for `Wrong Product` (**248 items | 13.8%**).
---

In [ ]:
import os
from matplotlib.ticker import FuncFormatter

OUT = "exported_charts_jpg"
os.makedirs(OUT, exist_ok=True)

def save_close(fname):
    plt.tight_layout()
    plt.savefig(f"{OUT}/{fname}", format="jpg", dpi=300, bbox_inches="tight")
    plt.close()

# 1. Order Funnel & Payment Method
fig, axes = plt.subplots(2, 1, figsize=(15, 13))
total_orders = len(orders)

status_counts = orders["order_status"].value_counts()
sns.barplot(
    x=status_counts.index, y=status_counts.values, ax=axes[0],
    hue=status_counts.index, palette="viridis", legend=False, width=0.5
)
axes[0].set_title("Order Status Distribution", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Number of Orders")
axes[0].set_xlabel("Order Status")
axes[0].set_ylim(0, status_counts.max() * 1.15)

for p in axes[0].patches:
    val = int(p.get_height())
    pct = (val / total_orders) * 100
    axes[0].annotate(
        f"{val:,} ({pct:.1f}%)", (p.get_x() + p.get_width() / 2., val),
        ha="center", va="bottom", fontsize=10, fontweight="bold"
    )

pm_order = orders["payment_method"].value_counts().index
status_order = ["Completed", "Returned", "Cancelled", "Pending"]
status_colors = {
    "Completed": "#4c72b0", "Returned": "#dd8452",
    "Cancelled": "#c44e52", "Pending": "#8172b3"
}

pm_ct = pd.crosstab(orders["payment_method"], orders["order_status"]).loc[pm_order, status_order]
pm_pct = pm_ct.div(pm_ct.sum(axis=1), axis=0) * 100

bar_width = 0.35
pm_ct.plot(
    kind="bar", stacked=True, ax=axes[1],
    color=[status_colors[s] for s in status_order], width=bar_width
)
axes[1].set_title("Payment Method Popularity by Order Status", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Number of Orders")
axes[1].set_xlabel("Payment Method")
axes[1].set_ylim(0, pm_ct.sum(axis=1).max() * 1.18)
axes[1].set_xlim(-0.35, len(pm_order) - 0.15)
axes[1].tick_params(axis="x", rotation=90)
axes[1].legend(title="Order Status", bbox_to_anchor=(1.01, 1), loc="upper left")

callout_offsets = {"Returned": -1500, "Cancelled": 1200, "Pending": 4000}

for idx, payment in enumerate(pm_order):
    row_total = pm_ct.loc[payment].sum()
    row_share = (row_total / total_orders) * 100
    axes[1].text(
        idx, row_total + (pm_ct.sum(axis=1).max() * 0.015),
        f"{row_total:,} ({row_share:.1f}%)",
        ha="center", va="bottom", fontsize=9, fontweight="bold"
    )
    cum_height = 0
    for status in status_order:
        val = pm_ct.loc[payment, status]
        pct = pm_pct.loc[payment, status]
        mid_y = cum_height + val / 2
        box_color = status_colors[status]
        if status == "Completed":
            axes[1].text(
                idx, mid_y, f"{val:,}\n({pct:.1f}%)",
                ha="center", va="center", fontsize=8.5, color="white", fontweight="bold"
            )
        else:
            axes[1].annotate(
                f"{val:,} ({pct:.1f}%)",
                xy=(idx + bar_width / 2, mid_y),
                xytext=(idx + bar_width / 2 + 0.06, mid_y + callout_offsets[status]),
                ha="left", va="center", fontsize=7.5, color="white", fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.22", fc=box_color, ec="none", alpha=0.95),
                arrowprops=dict(arrowstyle="-", color=box_color, lw=1.2)
            )
        cum_height += val

save_close("01_order_funnel_and_payment.jpg")

# 2. Currency & Monthly Trend
fig, axes = plt.subplots(2, 1, figsize=(16, 13))
total_comp = len(completed_orders)

curr_counts = completed_orders["currency"].value_counts()
sns.barplot(
    x=curr_counts.index, y=curr_counts.values, ax=axes[0],
    hue=curr_counts.index, palette="Blues_r", legend=False, width=0.5
)
axes[0].set_title("Completed Orders by Currency", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Completed Orders")
axes[0].set_xlabel("Currency")
axes[0].set_ylim(0, curr_counts.max() * 1.15)
axes[0].tick_params(axis="x", rotation=90)

for p in axes[0].patches:
    val = int(p.get_height())
    pct = (val / total_comp) * 100
    axes[0].annotate(
        f"{val:,} ({pct:.1f}%)", (p.get_x() + p.get_width() / 2., val),
        ha="center", va="bottom", fontsize=9.5, fontweight="bold"
    )

usd_orders = completed_orders[completed_orders["currency"] == "USD"]
monthly_usd = usd_orders.groupby("order_year_month").agg(
    orders_count=("order_id", "count"),
    net_sales=("net_sales_order", "sum"),
    profit=("profit_order", "sum")
).reset_index()

total_usd_sales = monthly_usd["net_sales"].sum()
total_usd_profit = monthly_usd["profit"].sum()
avg_usd_sales = monthly_usd["net_sales"].mean()
avg_usd_profit = monthly_usd["profit"].mean()

axes[1].plot(
    monthly_usd["order_year_month"], monthly_usd["net_sales"],
    marker="o", label="Net Sales (USD)", color="#1f77b4", linewidth=2
)
axes[1].plot(
    monthly_usd["order_year_month"], monthly_usd["profit"],
    marker="s", label="Profit (USD)", color="#2ca02c", linewidth=2
)
axes[1].axhline(
    avg_usd_sales, color="#1f77b4", linestyle="--", alpha=0.5,
    label=f"Avg Net Sales (${avg_usd_sales/1e6:.2f}M)"
)
axes[1].axhline(
    avg_usd_profit, color="#2ca02c", linestyle="--", alpha=0.5,
    label=f"Avg Profit (${avg_usd_profit/1e6:.2f}M)"
)

for idx, lbl, off_y in [(monthly_usd["net_sales"].idxmax(), "Peak", 14),
                        (monthly_usd["net_sales"].idxmin(), "Low", -24)]:
    val = monthly_usd.loc[idx, "net_sales"]
    pct = (val / total_usd_sales) * 100
    axes[1].annotate(
        f"{lbl}: ${val/1e6:.2f}M ({pct:.1f}%)", xy=(idx, val),
        xytext=(0, off_y), textcoords="offset points", ha="center",
        fontsize=8.5, fontweight="bold", color="white",
        bbox=dict(boxstyle="round,pad=0.25", fc="#1f77b4", ec="none")
    )

for idx, lbl, off_y in [(monthly_usd["profit"].idxmax(), "Peak", 14),
                        (monthly_usd["profit"].idxmin(), "Low", -24)]:
    val = monthly_usd.loc[idx, "profit"]
    pct = (val / total_usd_profit) * 100
    axes[1].annotate(
        f"{lbl}: ${val/1e3:.0f}K ({pct:.1f}%)", xy=(idx, val),
        xytext=(0, off_y), textcoords="offset points", ha="center",
        fontsize=8.5, fontweight="bold", color="white",
        bbox=dict(boxstyle="round,pad=0.25", fc="#2ca02c", ec="none")
    )

axes[1].set_title("Monthly Net Sales & Profit Trend (USD - Completed Orders)", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Amount (USD)")
axes[1].set_xlabel("Year-Month")
axes[1].set_ylim(monthly_usd["profit"].min() * 0.65, monthly_usd["net_sales"].max() * 1.15)
axes[1].yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"${x/1e6:.2f}M"))
axes[1].tick_params(axis="x", rotation=90)
axes[1].legend(title="Metric", loc="upper left", ncol=2)
axes[1].grid(True, linestyle="--", alpha=0.4)

save_close("02_currency_and_monthly_trend.jpg")

# 3. Product Category Performance & Profitability
usd_items = completed_items[completed_items["currency"] == "USD"].copy()
cat_perf = usd_items.groupby("product_category").agg(
    net_sales=("net_sales_item", "sum"),
    profit=("profit_item", "sum")
).sort_values("net_sales", ascending=False)

cat_perf["sales_share_pct"] = (cat_perf["net_sales"] / cat_perf["net_sales"].sum()) * 100
cat_perf["margin_pct"] = (cat_perf["profit"] / cat_perf["net_sales"]) * 100

fig, axes = plt.subplots(2, 1, figsize=(15, 13))

sns.barplot(
    x=cat_perf.index, y=cat_perf["net_sales"], ax=axes[0],
    hue=cat_perf.index, palette="crest", legend=False, width=0.55
)
axes[0].set_title("USD Net Sales by Product Category (Completed Items)", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Net Sales (USD)")
axes[0].set_xlabel("Product Category")
axes[0].set_ylim(0, cat_perf["net_sales"].max() * 1.18)
axes[0].yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"${x/1e6:.1f}M"))
axes[0].tick_params(axis="x", rotation=90)

for idx, p in enumerate(axes[0].patches):
    val = p.get_height()
    pct = cat_perf["sales_share_pct"].iloc[idx]
    axes[0].annotate(
        f"${val/1e6:.2f}M\n({pct:.1f}%)", (p.get_x() + p.get_width() / 2., val),
        ha="center", va="bottom", fontsize=8.5, fontweight="bold"
    )

sns.barplot(
    x=cat_perf.index, y=cat_perf["profit"], ax=axes[1],
    hue=cat_perf.index, palette="flare", legend=False, width=0.55
)
axes[1].set_title("USD Total Profit & Profit Margin % by Product Category", fontsize=13, fontweight="bold")
axes[1].set_ylabel("Total Profit (USD)")
axes[1].set_xlabel("Product Category")
axes[1].set_ylim(0, cat_perf["profit"].max() * 1.20)
axes[1].yaxis.set_major_formatter(FuncFormatter(lambda x, _: f"${x/1e6:.1f}M"))
axes[1].tick_params(axis="x", rotation=90)

for idx, p in enumerate(axes[1].patches):
    val = p.get_height()
    margin = cat_perf["margin_pct"].iloc[idx]
    axes[1].annotate(
        f"${val/1e6:.2f}M\n(Margin: {margin:.1f}%)", (p.get_x() + p.get_width() / 2., val),
        ha="center", va="bottom", fontsize=8, fontweight="bold"
    )

save_close("03_category_sales_and_profit.jpg")

# 4. Customer Segment, Age Group & Sales Channel
bins = [17, 25, 35, 45, 55, 65, 100]
labels = ["18-25", "26-35", "36-45", "46-55", "56-65", "65+"]
completed_orders["age_group"] = pd.cut(completed_orders["customer_age"], bins=bins, labels=labels)

seg_counts = completed_orders["customer_segment"].value_counts()
age_counts = completed_orders["age_group"].value_counts().reindex(labels)
ch_counts = completed_orders["sales_channel"].value_counts()

demo_plots = [
    (seg_counts, "Completed Orders by Customer Segment", "Customer Segment", "Set2", 0.45, "04a_customer_segment.jpg"),
    (age_counts, "Completed Orders by Age Group", "Age Group", "Blues_d", 0.50, "04b_customer_age_group.jpg"),
    (ch_counts, "Completed Orders by Sales Channel", "Sales Channel", "Purples_r", 0.45, "04c_sales_channel.jpg")
]

for series, title, xlabel, pal, w, fname in demo_plots:
    plt.figure(figsize=(11, 5.5))
    ax = sns.barplot(x=series.index, y=series.values, hue=series.index, palette=pal, legend=False, width=w)
    plt.title(title, fontsize=14, fontweight="bold")
    plt.ylabel("Completed Orders")
    plt.xlabel(xlabel)
    plt.ylim(0, series.max() * 1.15)
    plt.xticks(rotation=90)
    for p in ax.patches:
        val = int(p.get_height())
        pct = (val / total_comp) * 100
        ax.annotate(
            f"{val:,} ({pct:.1f}%)", (p.get_x() + p.get_width() / 2., val),
            ha="center", va="bottom", fontsize=10, fontweight="bold"
        )
    save_close(fname)

# 5. Operations, Ratings & Return Heatmap
ship_ct = pd.crosstab(completed_orders["shipping_method"], completed_orders["delivery_status"])
ship_pct = ship_ct.div(ship_ct.sum(axis=1), axis=0) * 100
deliv_order = ship_pct.columns.tolist()

palette_map = {
    s: ("#2A9D8F" if "early" in str(s).lower() else "#457B9D" if "time" in str(s).lower() else "#E63946")
    for s in deliv_order
}
bar_colors = [palette_map[s] for s in deliv_order]

ax = ship_pct.plot(kind="barh", stacked=True, figsize=(11, 5), color=bar_colors, width=0.55)
plt.title("Delivery Status by Shipping Method", fontsize=13, fontweight="bold")
plt.xlabel("Percentage (%)")
plt.ylabel("Shipping Method")
plt.xlim(0, 100)
plt.legend(title="Delivery Status", bbox_to_anchor=(1.01, 1), loc="upper left")
for i, method in enumerate(ship_pct.index):
    cum_pct = 0
    for status in deliv_order:
        pct = ship_pct.loc[method, status]
        cnt = ship_ct.loc[method, status]
        if pct >= 5:
            ax.text(
                cum_pct + pct / 2, i, f"{cnt:,} ({pct:.1f}%)",
                ha="center", va="center", fontsize=8.5, color="white", fontweight="bold"
            )
        cum_pct += pct
save_close("05a_delivery_status_by_shipping.jpg")

deliv_rating = completed_orders.groupby("delivery_status").agg(
    count=("order_id", "count"),
    mean_rating=("customer_rating", "mean")
).reindex(deliv_order)
total_deliv = deliv_rating["count"].sum()

plt.figure(figsize=(10, 5))
ax2 = sns.barplot(
    x=deliv_rating.index, y=deliv_rating["mean_rating"],
    hue=deliv_rating.index, palette=palette_map, legend=False, width=0.45
)
plt.title("Average Customer Rating by Delivery Status", fontsize=13, fontweight="bold")
plt.xlabel("Delivery Status")
plt.ylabel("Average Rating (1 to 5)")
plt.ylim(0, 5.8)
for i, p in enumerate(ax2.patches):
    rating_v = p.get_height()
    cnt_v = deliv_rating["count"].iloc[i]
    pct_v = (cnt_v / total_deliv) * 100
    ax2.annotate(
        f"{rating_v:.2f} / 5.0\n({cnt_v:,} | {pct_v:.1f}%)",
        (p.get_x() + p.get_width() / 2., rating_v),
        ha="center", va="bottom", fontsize=9, fontweight="bold"
    )
save_close("05b_rating_by_delivery_status.jpg")

returned_items = data[data["order_status"] == "Returned"].copy()
ret_matrix = pd.crosstab(returned_items["product_category"], returned_items["return_reason"])
ret_matrix_pct = ret_matrix.div(ret_matrix.sum(axis=1), axis=0) * 100
annot_matrix = ret_matrix.astype(str) + "\n(" + ret_matrix_pct.round(1).astype(str) + "%)"

plt.figure(figsize=(13, 7.5))
sns.heatmap(
    ret_matrix_pct, annot=annot_matrix, fmt="", cmap="Reds",
    cbar_kws={"label": "Share Within Category (%)"}, linewidths=1.5, linecolor="white"
)
plt.title("Return Reason by Product Category", fontsize=13, fontweight="bold")
plt.xlabel("Return Reason")
plt.ylabel("Product Category")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
save_close("05c_return_reason_heatmap.jpg")

print("Selesai! 9 JPG tersimpan di:", os.listdir(OUT))